In [1]:
!pip install --upgrade "optree>=0.13.0" diffusers transformers accelerate sentencepiece protobuf bitsandbytes
!git clone https://github.com/Sainava/Sai-ToMA.git
!pip install ftfy

import sys
sys.path.append('/kaggle/working/Sai-ToMA')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.6/457.6 kB 12.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 80.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 115.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.2 MB/s eta 0:00:0000:010:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstall

In [2]:
%%writefile token_shape_capture_hook.py
import torch
from pathlib import Path

class TokenShapeCaptureHook:
    def __init__(self, block_name: str, save_dir: str):
        self.block_name = block_name
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.call_count = 0
        self.log_file = self.save_dir / "shape_log.txt"

    def __call__(self, module, input, output):
        # New Diffusers FLUX single blocks return:
        # output = (encoder_hidden_states, hidden_states)
        if isinstance(output, tuple) and len(output) == 2:
            text_tensor = output[0]
            image_tensor = output[1]
        else:
            text_tensor = None
            image_tensor = output

        # Save image token matrix
        image_tensor_cpu = image_tensor.detach().cpu()
        image_matrix_path = self.save_dir / f"{self.block_name}_image_step_{self.call_count}.pt"
        torch.save(image_tensor_cpu, image_matrix_path)

        image_shape = list(image_tensor.shape)
        image_seq_len = image_tensor.shape[1] if len(image_tensor.shape) > 1 else None

        # Save text token matrix, if available
        if text_tensor is not None:
            text_tensor_cpu = text_tensor.detach().cpu()
            text_matrix_path = self.save_dir / f"{self.block_name}_text_step_{self.call_count}.pt"
            torch.save(text_tensor_cpu, text_matrix_path)

            text_shape = list(text_tensor.shape)
            text_seq_len = text_tensor.shape[1] if len(text_tensor.shape) > 1 else None
        else:
            text_matrix_path = None
            text_shape = None
            text_seq_len = None

        with open(self.log_file, "a") as f:
            f.write(
                f"block={self.block_name} | "
                f"call_count={self.call_count} | "
                f"image_shape={image_shape} | image_seq_len={image_seq_len} | "
                f"text_shape={text_shape} | text_seq_len={text_seq_len} | "
                f"image_matrix={image_matrix_path} | "
                f"text_matrix={text_matrix_path}\\n"
            )

        self.call_count += 1

Writing token_shape_capture_hook.py


In [3]:
%%writefile token_shape_capture_hook.py
import torch
import torch.nn as nn
from pathlib import Path

class TokenShapeCaptureHook:
    def __init__(self, block_name: str, save_dir: str):
        self.block_name = block_name
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.call_count = 0
        self.log_file = self.save_dir / "shape_log.txt"

    def __call__(self, module, input, output):
        out_tensor = output[0] if isinstance(output, tuple) else output
        shape = out_tensor.shape
        seq_len = shape[1] if len(shape) > 1 else None

        torch.save(out_tensor.detach().cpu(), self.save_dir / f"{self.block_name}_step_{self.call_count}.pt")
        with open(self.log_file, "a") as f:
            f.write(f"call_count={self.call_count} | shape={list(shape)} | seq_len={seq_len}\n")
        self.call_count += 1

Overwriting token_shape_capture_hook.py


In [ ]:
import os
import torch
import shutil
from diffusers import FluxPipeline
from diffusers.quantizers import PipelineQuantizationConfig
from hidden_state_capture_hook import HiddenStateCaptureHook
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import logging

logging.getLogger("diffusers").setLevel(logging.ERROR)

def run_baseline():
    # 1. Securely fetch your Hugging Face token
    try:
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception as e:
        print("Error: Could not find HF_TOKEN. Ensure the secret is attached to this notebook in the Add-ons menu!")
        return
    
    # 2. Globally login to Hugging Face (Prevents 401 Unauthorized errors)
    print("Authenticating with Hugging Face...")
    login(token=hf_token)
    
    model_id = "black-forest-labs/FLUX.1-schnell" 
    print(f"Loading Baseline {model_id}...")

    # Set up 4-bit quantization to fit FLUX in Kaggle's T4 GPUs
    quant_config = PipelineQuantizationConfig(
        quant_backend="bitsandbytes_4bit",
        quant_kwargs={"bnb_4bit_compute_dtype": torch.bfloat16, "bnb_4bit_quant_type": "nf4"}
    )
    
    # 3. Load the pipeline 
    pipe = FluxPipeline.from_pretrained(
        model_id, 
        quantization_config=quant_config, 
        torch_dtype=torch.bfloat16
    )
    
    # 4. Use CPU offload to manage memory safely with 4-bit quantization
    pipe.enable_model_cpu_offload()

    target_block_indices = [2, 10, 18]
    output_dir = "./flux_baseline_results"
    hooks, handles = [], []

    # 5. Attach the Hidden State Capture Hooks
    print("Attaching hooks...")
    for idx in target_block_indices:
        hook = HiddenStateCaptureHook(block_name=f"flux_block_{idx}", save_dir=output_dir)
        hooks.append(hook)
        target_block = getattr(pipe.transformer, 'single_transformer_blocks', pipe.transformer.transformer_blocks)[idx]
        handles.append(target_block.register_forward_hook(hook))
    
    prompts = [
        "A red apple on a white table",
        "A high tech neural network visualization, highly detailed, cyberpunk style",
        "The concept of time dissolving into a chaotic vortex"
    ]

    # 6. Generate and save the outputs
    for i, prompt in enumerate(prompts):
        print(f"\nProcessing Baseline Prompt {i+1}/3: '{prompt}'")
        
        # Catch the generated image in a variable
        image = pipe(prompt=prompt, num_inference_steps=10, guidance_scale=0.0).images[0]
        
        # Save the image as a PNG
        filename = f"flux_baseline_image_{i+1}.png"
        image.save(filename)
        print(f"✅ Saved {filename} to /kaggle/working/")
    
    # 7. Clean up and zip the captured tensors
    for handle in handles: handle.remove()
    shutil.make_archive("flux_baseline_results", 'zip', output_dir)
    print("\n✅ Baseline tensor zip created successfully! (flux_baseline_results.zip)")

run_baseline()

In [ ]:

    # Force Python and PyTorch to empty the garbage
import gc
gc.collect()
torch.cuda.empty_cache()
print("GPU Memory wiped and ready for the next run!")

In [4]:
import os
import torch
import shutil
import logging
import types

from diffusers import FluxPipeline
from diffusers.quantizers import PipelineQuantizationConfig
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

from toma.patch import apply_patch
from toma.flux_scheduler import FluxScheduler
from token_shape_capture_hook import TokenShapeCaptureHook

logging.getLogger("diffusers").setLevel(logging.ERROR)


def adapt_toma_single_blocks_for_new_diffusers(pipe):
    """
    Fixes ToMA FLUX single-block API mismatch with newer Diffusers.

    New Diffusers passes:
        hidden_states = image tokens
        encoder_hidden_states = text tokens

    Older ToMA expects:
        hidden_states = concatenated [text tokens, image tokens]

    This adapter concatenates text + image tokens before ToMA,
    then splits them back after ToMA.
    """

    for idx, block in enumerate(pipe.transformer.single_transformer_blocks):
        old_forward = block.forward

        def compatible_forward(
            self,
            hidden_states,
            encoder_hidden_states=None,
            temb=None,
            image_rotary_emb=None,
            joint_attention_kwargs=None,
            *args,
            _old_forward=old_forward,
            _idx=idx,
            **kwargs,
        ):
            # Old style call
            if encoder_hidden_states is None:
                try:
                    return _old_forward(
                        hidden_states=hidden_states,
                        temb=temb,
                        image_rotary_emb=image_rotary_emb,
                        joint_attention_kwargs=joint_attention_kwargs,
                    )
                except TypeError:
                    return _old_forward(
                        hidden_states,
                        temb,
                        image_rotary_emb,
                        joint_attention_kwargs,
                    )

            text_seq_len = encoder_hidden_states.shape[1]

            # Combine text + image token streams
            combined_hidden_states = torch.cat(
                [encoder_hidden_states, hidden_states],
                dim=1,
            )

            try:
                out = _old_forward(
                    hidden_states=combined_hidden_states,
                    temb=temb,
                    image_rotary_emb=image_rotary_emb,
                    joint_attention_kwargs=joint_attention_kwargs,
                )
            except TypeError:
                out = _old_forward(
                    combined_hidden_states,
                    temb,
                    image_rotary_emb,
                    joint_attention_kwargs,
                )

            if isinstance(out, tuple):
                if len(out) == 2:
                    return out
                out = out[0]

            # Split back into text tokens and image tokens
            new_encoder_hidden_states = out[:, :text_seq_len]
            new_hidden_states = out[:, text_seq_len:]

            return new_encoder_hidden_states, new_hidden_states

        block.forward = types.MethodType(compatible_forward, block)

    print("✅ Adapted ToMA single_transformer_blocks for new Diffusers API")


def find_flux_config():
    possible_config_paths = [
        "./config/flux.yaml",
        "/kaggle/working/Sai-ToMA/config/flux.yaml",
    ]

    for path in possible_config_paths:
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        "Could not find flux.yaml. Expected one of: "
        "./config/flux.yaml or /kaggle/working/Sai-ToMA/config/flux.yaml"
    )


def make_flux_scheduler(num_inference_steps, config_path):
    recompute_step = list(range(0, num_inference_steps))
    merge_step = list(range(0, num_inference_steps, 3))

    scheduler = FluxScheduler(
        timesteps=num_inference_steps,
        dst_recompute_timesteps=recompute_step,
        attn_recompute_timesteps=recompute_step,
        merge_step=merge_step,
        config_path=config_path,
    )

    # Safety wrapper:
    # Sometimes after scheduler exhaustion, ToMA returns bool instead of tuple.
    old_step_for_image = scheduler.step_for_image
    old_step_for_text = scheduler.step_for_text

    def safe_step_for_image():
        out = old_step_for_image()
        if isinstance(out, tuple):
            return out
        return False, True

    def safe_step_for_text():
        out = old_step_for_text()
        if isinstance(out, tuple):
            return out
        return False, True

    scheduler.step_for_image = safe_step_for_image
    scheduler.step_for_text = safe_step_for_text

    return scheduler


def inject_scheduler_into_toma(pipe, scheduler):
    """
    ToMA stores merge_scheduler inside _toma_info.
    Before every prompt, inject a fresh scheduler so it is not exhausted.
    """

    count = 0

    for module in pipe.transformer.modules():
        if hasattr(module, "_toma_info") and isinstance(module._toma_info, dict):
            module._toma_info["merge_scheduler"] = scheduler
            count += 1

        if hasattr(module, "processor"):
            processor = module.processor

            if hasattr(processor, "_toma_info") and isinstance(processor._toma_info, dict):
                processor._toma_info["merge_scheduler"] = scheduler
                count += 1

    print(f"✅ Fresh scheduler injected into {count} ToMA locations")


def run_toma_save_images_and_matrices():
    # -----------------------------
    # 1. Hugging Face authentication
    # -----------------------------
    try:
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        print("Error: Could not find HF_TOKEN. Ensure the secret is attached in Kaggle!")
        return

    print("Authenticating with Hugging Face...")
    login(token=hf_token)

    # -----------------------------
    # 2. Settings
    # -----------------------------
    model_id = "black-forest-labs/FLUX.1-schnell"

    num_inference_steps = 10
    ratio = 0.5
    dst_selection = "local_tile_wise_facility"
    num_tiles = 256
    merge_method = "attention"

    height = 512
    width = 512

    output_dir = "./toma_shape_results"

    config_path = find_flux_config()
    print(f"Using ToMA config: {config_path}")

    # -----------------------------
    # 3. Load FLUX pipeline
    # -----------------------------
    print(f"Loading model: {model_id}")

    quant_config = PipelineQuantizationConfig(
        quant_backend="bitsandbytes_4bit",
        quant_kwargs={
            "bnb_4bit_compute_dtype": torch.bfloat16,
            "bnb_4bit_quant_type": "nf4",
        },
    )

    pipe = FluxPipeline.from_pretrained(
        model_id,
        quantization_config=quant_config,
        torch_dtype=torch.bfloat16,
    )

    # -----------------------------
    # 4. Apply ToMA patch once
    # -----------------------------
    print("Applying ToMA Patch...")

    initial_scheduler = make_flux_scheduler(num_inference_steps, config_path)

    apply_patch(
        model=pipe,
        ratio=ratio,
        dst_selection=dst_selection,
        num_tiles=num_tiles,
        merge_method=merge_method,
        merge_scheduler=initial_scheduler,
    )

    adapt_toma_single_blocks_for_new_diffusers(pipe)

    print("✅ ToMA patch applied successfully")

    # -----------------------------
    # 5. Offload model safely
    # -----------------------------
    pipe.enable_model_cpu_offload()

    # -----------------------------
    # 6. Prepare output folder
    # -----------------------------
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    os.makedirs(output_dir, exist_ok=True)

    # -----------------------------
    # 7. Attach matrix capture hooks
    # -----------------------------
    target_block_indices = [2, 10, 18]

    handles = []

    print("Attaching matrix capture hooks...")

    for idx in target_block_indices:
        hook = TokenShapeCaptureHook(
            block_name=f"toma_block_{idx}",
            save_dir=output_dir,
        )

        target_block = pipe.transformer.single_transformer_blocks[idx]
        handle = target_block.register_forward_hook(hook)
        handles.append(handle)

        print(f"Hook attached to single_transformer_blocks[{idx}]")

    # -----------------------------
    # 8. Prompts
    # -----------------------------
    prompts = [
        "A red apple on a white table",
        "A high tech neural network visualization, highly detailed, cyberpunk style",
        "The concept of time dissolving into a chaotic vortex",
    ]

    # -----------------------------
    # 9. Run inference, save images + matrices
    # -----------------------------
    for i, prompt in enumerate(prompts):
        print(f"\nToMA prompt {i + 1}/{len(prompts)}: {prompt}")

        # Critical:
        # Create fresh scheduler for every prompt
        fresh_scheduler = make_flux_scheduler(num_inference_steps, config_path)
        inject_scheduler_into_toma(pipe, fresh_scheduler)

        generator = torch.Generator(device="cpu").manual_seed(42 + i)

        result = pipe(
            prompt=prompt,
            height=height,
            width=width,
            num_inference_steps=num_inference_steps,
            guidance_scale=0.0,
            max_sequence_length=512,
            generator=generator,
            output_type="pil",
        )

        image = result.images[0]

        image_path = os.path.join(output_dir, f"toma_prompt_{i + 1}.png")
        image.save(image_path)

        print(f"✅ Image saved: {image_path}")
        print(f"✅ Matrices captured for prompt {i + 1}")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # -----------------------------
    # 10. Remove hooks
    # -----------------------------
    for handle in handles:
        handle.remove()

    print("Hooks removed.")

    # -----------------------------
    # 11. Zip all results
    # -----------------------------
    zip_name = "toma_shape_results"

    if os.path.exists(zip_name + ".zip"):
        os.remove(zip_name + ".zip")

    shutil.make_archive(zip_name, "zip", output_dir)

    print("\n✅ ToMA results zip created successfully!")
    print("Saved as: toma_shape_results.zip")
    print("Includes:")
    print("- generated PNG images")
    print("- image token matrices (.pt)")
    print("- text token matrices (.pt)")
    print("- shape_log.txt")


run_toma_save_images_and_matrices()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Authenticating with Hugging Face...
Using ToMA config: /kaggle/working/Sai-ToMA/config/flux.yaml
Loading model: black-forest-labs/FLUX.1-schnell


model_index.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

✅ Image saved: ./toma_shape_results/toma_prompt_1.png
✅ Matrices captured for prompt 1

ToMA prompt 2/3: A high tech neural network visualization, highly detailed, cyberpunk style
✅ Fresh scheduler injected into 77 ToMA locations


  0%|          | 0/10 [00:00<?, ?it/s]

All blocks and repetitions have been processed.
✅ Image saved: ./toma_shape_results/toma_prompt_2.png
✅ Matrices captured for prompt 2

ToMA prompt 3/3: The concept of time dissolving into a chaotic vortex
✅ Fresh scheduler injected into 77 ToMA locations


  0%|          | 0/10 [00:00<?, ?it/s]

All blocks and repetitions have been processed.
✅ Image saved: ./toma_shape_results/toma_prompt_3.png
✅ Matrices captured for prompt 3
Hooks removed.

✅ ToMA results zip created successfully!
Saved as: toma_shape_results.zip
Includes:
- generated PNG images
- image token matrices (.pt)
- text token matrices (.pt)
- shape_log.txt
